# DOAgent — Push demo

This notebook runs the **push** scenario step by step: two agents (adversary and goal-seeker) in a PettingZoo MPE push environment. It only uses the `doagent` library and code defined here. Run in Google Colab.

**What you'll do:** Install doagent + PettingZoo → create a session with file as the shared data model → define policies and a push env wrapper → run the loop → run analysis (provenance, traceability, interpretability).

## Step 1 — Install the library and PettingZoo

Install DOAgent and the PettingZoo MPE dependencies needed for the push environment.

In [ ]:
!pip install -q git+https://github.com/cabrerac/doagent.git
!pip install -q pettingzoo[mpe] mpe2 pygame

## Step 2 — Imports

Import doagent and the MPE push environment. We will wrap the PettingZoo env so its step return format matches what DOAgent expects.

In [ ]:
import random
from typing import Any, Dict

from doagent import Session, RunReporter, make_env
from doagent.analysis import interpretability, provenance, traceability

## Step 3 — Wrap PettingZoo for DOAgent

PettingZoo's `parallel_env` returns `(observations, rewards, terminations, truncations, infos)` from `step`. DOAgent expects a dict with `observations`, `rewards`, and `terminations` (or `done`). This wrapper converts between the two.

In [ ]:
def create_push_env(max_cycles=100, render_mode=None, **kwargs):
    from mpe2.simple_push_v3 import parallel_env
    params = {"max_cycles": max_cycles, "continuous_actions": False, "dynamic_rescaling": False, **kwargs}
    if render_mode:
        params["render_mode"] = render_mode
    raw = parallel_env(**params)

    class Wrapper:
        agents = getattr(raw, "agents", None) or getattr(raw, "possible_agents", [])
        _raw = raw
        def reset(self, *, seed=None):
            obs, _ = raw.reset(seed=seed)
            return dict(obs)
        def step(self, actions):
            obs, rewards, terms, truncs, infos = raw.step(actions)
            return {"observations": dict(obs), "rewards": dict(rewards), "terminations": dict(terms)}
        def render(self):
            out = raw.render()
            return out  # numpy array when render_mode='rgb_array', else None
    return Wrapper()

## Step 4 — Define policies

Two heuristic policies: goal-seek (moves toward goal from observation) and push-block (moves toward block). Both use epsilon-greedy exploration.

In [ ]:
def _action_from_vector(dx: float, dy: float) -> int:
    if abs(dx) < 1e-6 and abs(dy) < 1e-3:
        return 0
    if abs(dx) >= abs(dy):
        return 2 if dx > 0 else 1
    return 4 if dy > 0 else 3

def _epsilon_greedy(base: int, epsilon: float, rng: random.Random) -> int:
    return rng.choice([0, 1, 2, 3, 4]) if rng.random() < epsilon else base

def heuristic_goal_seek(params: Dict[str, Any]):
    epsilon = float(params.get("epsilon", 0.0))
    rng = random.Random(params.get("seed", 0))
    def decide(request):
        obs = request.get("inputs", {}).get("observation", [])
        dx, dy = (float(obs[2]), float(obs[3])) if len(obs) >= 4 else (0.0, 0.0)
        return {"decision": {"action": _epsilon_greedy(_action_from_vector(dx, dy), epsilon, rng)}}
    return decide

def heuristic_push_block(params: Dict[str, Any]):
    epsilon = float(params.get("epsilon", 0.0))
    rng = random.Random(params.get("seed", 0))
    def decide(request):
        obs = request.get("inputs", {}).get("observation", [])
        dx, dy = (float(obs[6]), float(obs[7])) if len(obs) >= 8 else (0.0, 0.0)
        return {"decision": {"action": _epsilon_greedy(_action_from_vector(dx, dy), epsilon, rng)}}
    return decide

## Step 5 — Configure session with file as the shared data model

The session is configured with **file as the shared data model**: records (outcomes, agent_updates, traces) are written to a run folder under `output_base`, so we can run analysis by `run_id` later. **Decentralisation:** we do not set a custom topology here, so the default is centralised (all agents see all records); you can set `topology: { mode: "peer_to_peer", visibility: {...} }` to restrict which records each agent sees. **Openness:** you provide the environment, policies, and run loop; the library provides the session, records all decisions and outcomes via the shared data model, and exposes analysis interfaces (provenance, traceability, interpretability).

In [ ]:
output_base = "./output"
session = Session.from_config({
    "shared_data": {"type": "file"},
    "scenario_name": "push",
    "output_base": output_base,
    "run_config": {"logging_level": 2},
    "policies": {"heuristic_goal_seek": heuristic_goal_seek, "heuristic_push_block": heuristic_push_block},
})
print(f"Run id: {session.run_id}")

## Step 6 — Create env and agents, then run the loop

Parameters match **examples/push_demo** (100 rounds, seed 123, same agent configs). We create the env with `render_mode="rgb_array"` so we can display frames in the notebook. Every 25 rounds we show the current frame; the final frame is shown at the end.

In [ ]:
rounds, seed = 100, 123
render_every = 25  # show frame every N rounds (set to 0 to disable)
env = make_env(create_push_env, max_cycles=rounds, continuous_actions=False, dynamic_rescaling=False, render_mode="rgb_array")
wrapped = session.wrap_env(env, env_actor="push_env")
agents = session.create_agents([
    {"id": "adversary_0", "policy": {"name": "heuristic_push_block", "params": {"epsilon": 0.2, "seed": 1}}},
    {"id": "agent_0", "policy": {"name": "heuristic_goal_seek", "params": {"epsilon": 0.2, "seed": 2}}},
], goal="push_towards_landmark")

import matplotlib.pyplot as plt
from IPython.display import display, clear_output

observations = wrapped.reset(seed=seed)
for round_id in range(1, rounds + 1):
    actions = {aid: agents[aid].decide(observations.get(aid, {}), round_id)["action"] for aid in agents}
    step = wrapped.step(actions)
    observations = step["observations"]
    if render_every and round_id % render_every == 0:
        frame = wrapped.render()
        if frame is not None:
            fig, ax = plt.subplots(1, 1, figsize=(5, 5))
            ax.imshow(frame)
            ax.set_title(f"Push env — round {round_id}")
            ax.axis("off")
            plt.show()
frame = wrapped.render()
if frame is not None:
    fig, ax = plt.subplots(1, 1, figsize=(5, 5))
    ax.imshow(frame)
    ax.set_title(f"Push env — final (round {rounds})")
    ax.axis("off")
    plt.show()
print(f"Completed {rounds} rounds.")

## Step 7 — Run analysis (provenance, traceability, interpretability)

Use `doagent.analysis` with `write_output=True` to write artefacts under `output/<run_id>/analysis/`. We use the effective id from provenance for interpretability so explanations refer to the same outcome.

In [ ]:
run_id = session.run_id
effective_id = provenance.render_chain_tree("last", run_id, output_base=output_base, write_output=True)
traceability.build_trace_graph(run_id, output_base=output_base, write_output=True)
last_id = effective_id or "last"
explanations = interpretability.get_explanations_for(last_id, run_id, output_base=output_base, write_output=True)
print(f"Analysis written to {output_base}/{run_id}/analysis/")
print(f"Explanations for last outcome: {len(explanations)} records.")

## Step 8 — View and interpret the analysis results

The analysis step wrote figures and JSON under `output/<run_id>/analysis/`. Each block below shows one output and a short explanation.

#### Provenance tree

Shows the chain of records that led to the last outcome—which decisions and environment steps produced the final state. Useful to see "how we got here."

**How to read it:**
- **Node colors:** outcome (light blue), agent_update (light green), trace (gold), initial_state (dark).
- **Arrows:** derived_from (blue), trace_to (red), enabled_by (green), from (orange). Arrows point from cause to effect.

In [ ]:
from pathlib import Path
from IPython.display import Image, display

base = Path(output_base) / run_id / "analysis"
provenance_png = base / "provenance" / "provenance_tree.png"
if provenance_png.exists():
    display(Image(filename=str(provenance_png)))
else:
    print("Provenance tree not found (re-run Step 7).")

#### Trace graph

Cause–effect links between records (who acted, what they observed, what changed). Answers "which actions led to this outcome?"

**How to read it:**
- **Nodes:** initial state (dark), regular states (light blue), dedup convergence (gold). Labels like "r3" = round 3.
- **Edges:** each arrow is colored by the agent who caused that transition (e.g. agent_0 blue, agent_1 orange).

In [ ]:
trace_png = base / "traceability" / "trace_graph.png"
if trace_png.exists():
    display(Image(filename=str(trace_png)))
else:
    print("Trace graph not found (re-run Step 7).")

#### Explanations

Records that explain the chosen outcome (e.g. the last step). Full JSON below.

**JSON fields (each record):**
- **id**: record id. **kind**: `outcome`, `agent_update`, or `explanation`.
- **actor**: who produced it (e.g. agent id or `push_env`). **timestamp**: when.
- **payload**: content (e.g. observations, actions, rewards). **provenance** / **accountability**: optional links.
- **_role**: optional—`outcome` (env step), `decision` (agent_update that led to it), or `explanation`.

In [ ]:
import json
expl_path = base / "interpretability" / "explanations_for_last.json"
if expl_path.exists():
    with open(expl_path, encoding="utf-8") as f:
        data = json.load(f)
    print(json.dumps(data, indent=2, default=str))
else:
    print("Explanations file not found (re-run Step 7).")

---
**Next:** Try the [Grid-world demo](03_gridworld_demo.ipynb) for a discovery scenario with causal attribution.